In [0]:
# --------------------------------------------------
# Pure Bronze ingestion for all 7 e-commerce datasets
# --------------------------------------------------
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_ingestion_uc")

# --------------------------------------------------
# Datasets configuration
# --------------------------------------------------
datasets = {
    "customers": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/customers_dataset",
    "order_items": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/order_items_dataset",
    "order_payments": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/order_payment_dataset",
    "orders": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/orders_dataset",
    "products": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/products_dataset",
    "product_category_translation": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/product_category_name_translation",
    "sellers": "s3://ecommerce-analytics-datalake1/processed/ecommerce_sales_parquet/sellers_dataset"
}

BRONZE_SCHEMA = "ecommerce_catalog.bronze"

# --------------------------------------------------
# Bronze ingestion pipeline
# --------------------------------------------------
for table_name, path in datasets.items():
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}_table"
    try:
        logger.info(f"Starting ingestion for {table_name} from {path}")

        # Check if source path exists
        try:
            files = dbutils.fs.ls(path)
        except Exception:
            logger.warning(f"Source path does not exist: {path}")
            files = []

        if len(files) == 0:
            logger.warning(f"No files found at {path}. Skipping ingestion.")
            continue

        # Read Parquet
        df = spark.read.format("parquet").load(path)

        # Optional: normalize column names to lowercase
        df = df.toDF(*[c.lower() for c in df.columns])

        # Ensure Bronze schema exists
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

        # Write as Delta table (overwrite mode, only original columns)
        df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(full_table_name)

        logger.info(f"Bronze table ingested successfully: {full_table_name}")

    except Exception as e:
        logger.error(f"Ingestion failed for {table_name}")
        logger.error(str(e))

logger.info("Bronze ingestion pipeline finished for all datasets")